# Tools in Agno

Tools are what transform an LLM into an **agent** — they give the model the ability to take actions and access real-world data.

Agno supports:
1. **Built-in tools** — DuckDuckGo, YFinance, Shell, File, Email, etc.
2. **Custom Python functions** — any `def` decorated with `@tool` or passed directly
3. **MCP tools** — any Model Context Protocol server
4. **Toolkits** — bundles of related tools (e.g., `YFinanceTools`)

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
for var in ['OPENAI_API_KEY', 'OPENAI_BASE_URL', 'OPENAI_API_BASE']:
    if os.getenv(var):
        del os.environ[var]

os.environ["OPENAI_API_KEY"] = os.getenv("OPEN_ROUTER_KEY")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

### 2a. Custom Tool — Simple Python Function
Any Python function can be a tool. Agno uses docstrings + type hints for the tool schema.

In [2]:
import random
from datetime import datetime
from agno.agent import Agent
from agno.models.openai import OpenAIChat

def get_weather(city: str) -> str:
    """Get the current weather for a given city.
    
    Args:
        city: The name of the city to get weather for.
    
    Returns:
        A string describing current weather conditions.
    """
    # Simulated weather data (replace with real API call)
    conditions = ["Sunny", "Cloudy", "Rainy", "Partly Cloudy"]
    temp = random.randint(15, 35)
    condition = random.choice(conditions)
    return f"{city}: {condition}, {temp}°C, Humidity 65%"

def get_current_time() -> str:
    """Get the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

weather_agent = Agent(
    name="Weather Agent",
    model=OpenAIChat(id="google/gemini-2.5-flash"),
    tools=[get_weather, get_current_time],
    instructions=[
        "Always check the current time before giving weather info.",
        "Provide helpful travel recommendations based on the weather.",
    ],
    
    markdown=True,
)

weather_agent.print_response(
    "What's the weather like in Mumbai and Chennai right now? Should I carry an umbrella?",
    stream=True,
)

Output()

### 2b. Built-in Tools
Agno ships with many production-ready toolkits.

In [3]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.duckduckgo import DuckDuckGoTools
from agno.tools.yfinance import YFinanceTools

financial_news_agent = Agent(
    name="Financial News Agent",
    model=OpenAIChat(id="google/gemini-2.5-flash"),
    tools=[
        DuckDuckGoTools(),
        YFinanceTools(
            enable_stock_price=True,
            enable_analyst_recommendations=True,
            enable_company_info=True,
            enable_company_news=True,
        ),
    ],
    instructions=[
        "Use tables to display stock data.",
        "Always include the current price and analyst consensus.",
        "Search for recent news about the company.",
    ],
    
    markdown=True,
)

financial_news_agent.print_response(
    "Give me a comprehensive analysis of Apple (AAPL) stock.",
    stream=True,
)

Output()

### 2c. Custom Toolkit Class
Group related tools into a `Toolkit` class for cleaner organization.

In [4]:
from agno.tools import Toolkit
from agno.agent import Agent
from agno.models.openai import OpenAIChat

class DatabaseTools(Toolkit):
    """Simulated database access toolkit."""

    def get_user(self, user_id: str) -> str:
        """Retrieve user info from database by user ID."""
        # Simulated DB lookup
        users = {
            "U001": {"name": "Mali", "plan": "Pro", "joined": "2024-01"},
            "U002": {"name": "Raam", "plan": "Enterprise", "joined": "2023-06"},
        }
        user = users.get(user_id, {"name": "Unknown", "plan": "None"})
        return str(user)

    def get_order_status(self, order_id: str) -> str:
        """Get the status of an order by order ID."""
        statuses = {
            "ORD123": "Shipped — arriving in 2 days",
            "ORD456": "Processing — estimated dispatch tomorrow",
        }
        return statuses.get(order_id, "Order not found")

support_agent = Agent(
    name="Support Agent",
    model=OpenAIChat(id="google/gemini-2.5-flash"),
    tools=[DatabaseTools()],
    instructions=["Look up user and order details before responding."],
    
    markdown=True,
)

support_agent.print_response(
    "User U001 is asking about order ORD123.",
    stream=True,
)

Output()

### 2d. Tool Call Inspection
Agno makes it easy to see exactly which tools were called and with what arguments.

In [5]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.duckduckgo import DuckDuckGoTools

agent = Agent(
    model=OpenAIChat(id="google/gemini-2.5-flash"),
    tools=[DuckDuckGoTools()],
    markdown=True,
)

response = agent.run("What are the latest developments in  AI?")
print(response.content)
print(f"Response type: {type(response)}")
print(f"Messages count: {len(response.messages)}")

for i, msg in enumerate(response.messages):
    print(f"\n[{i}] type={type(msg).__name__}, role={getattr(msg, 'role', '?')}")
    print(f"     attrs: {[a for a in dir(msg) if not a.startswith('_')]}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"     tool_calls: {msg.tool_calls}")


Here's a summary of the latest AI developments:

**Major Developments:**

*   **Anthropic's IPO Filing and Call for Freeze:** Anthropic, the company behind the Claude chatbot, has filed for a massive $965 billion IPO. Interestingly, they have also called for a global freeze in AI development, warning of the risk of humans losing control, and offered to suspend work on more powerful systems if others would do the same.
*   **New Technical Upgrades:** OpenAI and Microsoft have released new technical upgrades to their AI systems.
*   **Infrastructure Investments:** Alphabet, among other companies, is making multi-billion dollar investments in AI infrastructure.

**Specific AI Model News:**

*   **Anthropic's Claude Opus 4.8:** Leaked details suggest Claude Opus 4.8 will enhance visual understanding and multi-step reasoning.
*   **OpenAI's GPT-5.6:** This model was also mentioned in a review of key AI developments for 2026.

**Regulatory Landscape:**

*   **EU AI Act:** The EU has establis